# Капстоун · ноутбук 2. Посттрейн: SFT и DPO

*Курс «Modern LLMs» · капстоун · редакция 2026.10.3*

> **Что здесь происходит.** Базовая модель из ноутбука 1 умеет продолжать текст,
> но не умеет вести диалог. Мы научим её формату — chat-шаблон, маскирование
> лосса по ответам ассистента, упаковка (M13) — а затем сдвинем предпочтения
> через DPO (M14).
>
> **Главный эксперимент — §5.** Мы намеренно забудем включить закрывающий
> `<|eot_id|>` в обучаемую часть маски и посмотрим, что получится. Измерим
> изменение цели обучения и поведение обеих ветвей.
>
> **Контракт Colab.** Ноутбук самодостаточен: если чекпойнта ноутбука 1 рядом
> нет, он обучит маленькую базовую модель сам. Сети не требуется.

Рабочее различие курса: SFT показывает желательное поведение, а обучение по предпочтениям
и RL используют другой обучающий сигнал. На нашем
масштабе проверяется механика и формат, а не frontier-качество ответа.

При первом проходе выполняйте ячейки по порядку и сопоставляйте код с пояснениями. Во втором измените один параметр, сохраните результаты и объясните, какое утверждение опыт позволяет проверить.

## 1. Окружение и базовая модель

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "пакет проекта: " + found

    try:
        import minillm  # noqa: F401
        return "minillm уже доступен"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Загрузите ZIP-пакет Capstone из урока Stepik. Структура каталогов будет восстановлена автоматически.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "архив проекта: " + str(archive_path)

    raise RuntimeError("minillm не найден. Скачайте языковой ZIP-пакет Capstone из урока Stepik и либо распакуйте его целиком, либо загрузите ZIP в текущую сессию Colab.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import os, torch, numpy as np, random

# Профиль забега. SMOKE рассчитан на CPU и укладывается в минуты; FULL имеет
# смысл только на GPU и даёт заметно более связные ответы.
PROFILE = "SMOKE"
BUDGET = {
    "SMOKE": dict(base_steps=250, sft_steps=200, sft_batch=4, seq_len=192,
                  dpo_steps=100, dpo_batch=4, hidden=128, layers=3, dialogues=800),
    "FULL":  dict(base_steps=2000, sft_steps=1500, sft_batch=8, seq_len=256,
                  dpo_steps=600, dpo_batch=8, hidden=384, layers=6, dialogues=6000),
}[PROFILE]
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    BUDGET = {**BUDGET, "base_steps": min(BUDGET["base_steps"], 10),
              "sft_steps": min(BUDGET["sft_steps"], 12),
              "dpo_steps": min(BUDGET["dpo_steps"], 8),
              "dialogues": min(BUDGET["dialogues"], 80),
              "sft_batch": min(BUDGET["sft_batch"], 2),
              "dpo_batch": min(BUDGET["dpo_batch"], 2)}
SEED = 0
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| устройство:", DEVICE)

Базовая модель. Если рядом лежит чекпойнт ноутбука 1 — берём его; иначе учим
маленькую базовую модель здесь же. Второй путь быстрее и достаточен для всех
выводов ноутбука: нас интересует, **чему учит маска**, а не абсолютное качество.

In [ ]:
from minillm.chat import CHAT_SPECIALS
from minillm.config import MiniLLMConfig
from minillm.data import TokenStream
from minillm.model import MiniLLM
from minillm.tokenizer import BBPETokenizer
from minillm.train import TrainConfig, train

RU_SLOTS = dict(
    subject=["языковая модель", "трансформер", "механизм внимания", "токенизатор",
             "оптимизатор", "нормализация", "остаточная связь", "роутер экспертов"],
    verb=["предсказывает", "обрабатывает", "сокращает", "ускоряет", "стабилизирует"],
    obj=["следующий токен", "длинный контекст", "стоимость шага", "объём памяти",
         "масштаб активаций", "нагрузку экспертов"],
)

def sentence(rng, slots):
    return " ".join([rng.choice(slots["subject"]), rng.choice(slots["verb"]),
                     rng.choice(slots["obj"])]).capitalize() + "."

rng = random.Random(SEED)
base_text = "\n".join(" ".join(sentence(rng, RU_SLOTS) for _ in range(4))
                      for _ in range(4000))

def usable_checkpoint(path: Path):
    """Чекпойнт годится, только если его словарь знает управляющие токены чата.

    Словарь без `<|eot_id|>` уронил бы ноутбук на первой же маске лосса, причём
    с невнятным KeyError. Ноутбук 1 резервирует эти идентификаторы заранее, но
    чекпойнт мог быть собран старой версией — поэтому проверяем явно.
    """
    if not (path / "config.json").exists():
        return None, None, "чекпойнта ноутбука 1 рядом нет"
    tok = BBPETokenizer.load(path / "tokenizer.json")
    missing = [s for s in CHAT_SPECIALS if s not in tok.specials]
    if missing:
        return None, None, f"в словаре чекпойнта нет токенов {missing}"
    from minillm.train import load_checkpoint
    model, _, _, _ = load_checkpoint(path, device=DEVICE)
    return model, tok, None

CHECKPOINT = Path("runs/pretrain/final")
base_model, tokenizer, reason = usable_checkpoint(CHECKPOINT)
if base_model is not None:
    print("загружен чекпойнт ноутбука 1:", base_model.num_parameters(), "параметров")
else:
    print(f"учим базовую модель сами: {reason}")
    tokenizer = BBPETokenizer.train(base_text, vocab_size=1024, specials=CHAT_SPECIALS)
    cfg = MiniLLMConfig.profile("S", vocab_size=tokenizer.vocab_size,
                                hidden_size=BUDGET["hidden"],
                                num_hidden_layers=BUDGET["layers"],
                                num_attention_heads=4, num_key_value_heads=2,
                                head_dim=32, max_position_embeddings=BUDGET["seq_len"])
    base_model = MiniLLM(cfg).to(DEVICE)
    stream = TokenStream.from_text(base_text, tokenizer, seq_len=128)
    print(f"параметров: {base_model.num_parameters():,}")
    train(base_model, stream, TrainConfig(steps=BUDGET["base_steps"], seq_len=128,
                                          micro_batch_size=8, optimizer="muon",
                                          seed=SEED, device=DEVICE, log_every=125))
print("словарь:", tokenizer.vocab_size)

## 2. Набор инструкций

Диалоги используют ту же слотовую грамматику, что и претрейн-корпус. Это уменьшает различие лексики и тем между стадиями, но не доказывает, что SFT учит только формату. Случайные сочетания слов здесь служат учебными данными и могут быть содержательно неверными. В опыте с EOT обе ветви получают одинаковые диалоги: так мы изолируем изменение маски, а не приписываем все эффекты SFT одной причине.

In [ ]:
from minillm.chat import ChatTemplate, Message, SFTDataset

# Формулировки подобраны так, чтобы согласование работало для подлежащих
# любого рода: «языковая модель», «трансформер», «остаточная связь».
QUESTIONS = [
    ("Что делает {s}?", "{s} {v} {o}."),
    ("Для чего служит {s}?", "{s} {v} {o}. Это и есть основная функция."),
    ("Как влияет {s} на модель?", "{s} {v} {o} и тем самым меняет поведение модели."),
]

def make_dialogues(n: int, seed: int = 0) -> list[list[Message]]:
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        question, answer = rng.choice(QUESTIONS)
        s = rng.choice(RU_SLOTS["subject"])
        v, o = rng.choice(RU_SLOTS["verb"]), rng.choice(RU_SLOTS["obj"])
        out.append([
            Message("system", "Ты отвечаешь кратко и по делу."),
            Message("user", question.format(s=s)),
            Message("assistant", answer.format(s=s.capitalize(), v=v, o=o)),
        ])
    return out

template = ChatTemplate()
dialogues = make_dialogues(BUDGET["dialogues"], seed=SEED)
print(template.render(dialogues[0], add_generation_prompt=False))

## 3. Маска: чему именно учится модель

Правило M13: учим ответы ассистента и **закрывающий `<|eot_id|>`**; не учим
системный промпт, реплики пользователя и заголовки ролей — их пишет рантайм.

In [ ]:
EOT_ID = tokenizer.specials["<|eot_id|>"]
PAD_ID = tokenizer.specials["<|begin_of_text|>"]

ids, mask = template.encode_with_mask(dialogues[0], tokenizer)
print(f"токенов: {len(ids)}, обучаемых: {sum(mask)} "
      f"({template.trainable_fraction(mask):.1%})\n")

print(f"{'токен':<22} {'id':>6}  учим")
for token_id, flag in list(zip(ids, mask)):
    piece = tokenizer.decode([token_id]).replace("\n", "\\n")
    print(f"{piece!r:<22} {token_id:>6}  {'ДА' if flag else '—'}")

In [ ]:
dataset, stats = SFTDataset.from_dialogues(dialogues, tokenizer, template,
                                           seq_len=BUDGET["seq_len"], pad_id=PAD_ID)
print("упаковка:", stats)
print(f"\nбез упаковки понадобилось бы {stats['dialogues']} строк по "
      f"{BUDGET['seq_len']} токенов,")
print(f"с упаковкой — {stats['bins']}: экономия ×{stats['packing_ratio']}, "
      f"padding {stats['padding_fraction']:.1%}")

Упаковка здесь корректностная, а не только экономическая. Вместе с `ids` и
`labels` набор данных возвращает `segment_ids`; модель строит блочно-диагональную
causal mask и сбрасывает RoPE-позиции внутри каждого диалога. Поэтому изменение
одного packed-примера не может повлиять на logits соседнего примера. `<|eot_id|>`
обозначает границу протокола, но сам по себе такой изоляции не обеспечил бы.

## 4. SFT

Обучаем ту же модель тем же циклом — меняется только источник батчей: вместо потока токенов приходит тройка `(ids, labels, segment_ids)`: −100
выключает прямой loss, а segment IDs изолируют независимые диалоги в attention.


Потеря SFT и проверочная потеря усредняются по обучаемым целям после причинного сдвига. При накоплении градиентов каждый микробатч получает вес, пропорциональный числу таких целей, чтобы изменение разбиения не меняло выбранную функцию потерь.

In [ ]:
import copy

sft_model = copy.deepcopy(base_model)
sft_cfg = TrainConfig(steps=BUDGET["sft_steps"], seq_len=BUDGET["seq_len"],
                      micro_batch_size=BUDGET["sft_batch"], optimizer="muon",
                      seed=SEED, device=DEVICE, log_every=BUDGET["sft_steps"] // 4,
                      out_dir="runs/sft")
sft_state = train(sft_model, dataset, sft_cfg)

In [ ]:
from minillm.generate import SamplingConfig, generate

def ask(model, question: str, max_new_tokens: int = 60, temperature: float = 0.0) -> tuple[str, int]:
    ids, _ = template.encode_with_mask(
        [Message("system", "Ты отвечаешь кратко и по делу."),
         Message("user", question)], tokenizer, add_generation_prompt=True)
    out, stats = generate(model, ids, max_new_tokens,
                          SamplingConfig(temperature=temperature, seed=SEED),
                          stop_tokens={EOT_ID})
    answer = tokenizer.decode(out[len(ids):])
    return answer, stats.generated_tokens

question = "Что делает токенизатор?"
for label, model in (("до SFT", base_model), ("после SFT", sft_model)):
    answer, n = ask(model, question)
    print(f"--- {label} ({n} токенов)\n{answer!r}\n")

## 5. Главный эксперимент: забытый `<|eot_id|>`

Повторяем SFT ровно с одним изменением: `train_eot=False`. Данные те же,
последовательности те же — отличается **одна позиция маски** на каждый диалог.

In [ ]:
broken_dataset, _ = SFTDataset.from_dialogues(dialogues, tokenizer, template,
                                              seq_len=BUDGET["seq_len"], pad_id=PAD_ID,
                                              train_eot=False)
a_ids, a_mask = template.encode_with_mask(dialogues[0], tokenizer, train_eot=True)
b_ids, b_mask = template.encode_with_mask(dialogues[0], tokenizer, train_eot=False)
diff = [i for i, (x, y) in enumerate(zip(a_mask, b_mask)) if x != y]
print(f"последовательности идентичны: {a_ids == b_ids}")
print(f"маски различаются в позициях {diff} — это {tokenizer.decode([a_ids[diff[0]]])!r}")

broken_model = copy.deepcopy(base_model)
train(broken_model, broken_dataset,
      TrainConfig(steps=BUDGET["sft_steps"], seq_len=BUDGET["seq_len"],
                  micro_batch_size=BUDGET["sft_batch"], optimizer="muon", seed=SEED,
                  device=DEVICE, log_every=BUDGET["sft_steps"], out_dir="runs/sft_broken"))

In [ ]:
eot_rows = []

print(f"{'модель':<16} {'p(<|eot_id|>) после ответа':>28} {'длина ответа':>14}")
for label, model in (("маска верная", sft_model), ("EOT забыт", broken_model)):
    ids, _ = template.encode_with_mask(dialogues[0], tokenizer)
    prefix = ids[:len(ids) - 1]        # всё, кроме закрывающего EOT
    with torch.no_grad():
        logits, _ = model(torch.tensor([prefix], device=DEVICE))
        p_eot = torch.softmax(logits[0, -1].float(), dim=-1)[EOT_ID].item()
    _, generated = ask(model, "Что делает токенизатор?", max_new_tokens=120)
    eot_rows.append(dict(label=label, probability=p_eot, generated_tokens=generated))
    print(f"{label:<16} {p_eot:>28.4f} {generated:>14}")

answer, n = ask(broken_model, "Что делает токенизатор?", max_new_tokens=120)
print(f"\nответ модели с забытым EOT ({n} токенов):\n{answer!r}")

# Use one correct mask for both models; the generator is a diagnostic fixture.
from minillm.train import evaluate
check_dialogues = make_dialogues(32, seed=SEED + 991)
check_dataset, _ = SFTDataset.from_dialogues(check_dialogues, tokenizer, template,
    seq_len=BUDGET["seq_len"], pad_id=PAD_ID, train_eot=True)
shared_scores = {name: evaluate(model, check_dataset, sft_cfg, batches=4)
                 for name, model in (("correct_mask", sft_model), ("omitted_eot", broken_model))}
assert sum(a_mask) == sum(b_mask) + len(diff)
print("EOT supervised positions:", sum(a_mask), "vs", sum(b_mask))
print("Shared correct-mask evaluation:", shared_scores)

Исключение EOT меняет и сумму потерь, и число обучаемых позиций в знаменателе среднего. Поэтому обучающие loss двух ветвей относятся к разным целям и не обязаны совпадать. Проверка с той же ошибочной маской может скрыть отсутствие обучения завершению ответа.

Обе модели оцениваются на одном наборе с корректной маской, включающей EOT. Отдельно смотрим вероятность EOT после заданного ответа и фактическую длину жадной генерации. Низкая вероятность одного токена и незавершившаяся генерация — связанные, но разные наблюдения. Короткий режим может ещё не дать резкого контраста; таблица должна показать его фактическую величину.

## 6. Пары предпочтений

Пары здесь строит авторский шаблон, без генерации моделью и без человеческой разметки. `chosen` содержит краткий ответ из `QUESTIONS` с подстановкой слов, а `rejected` добавляет два одинаковых предложения-заполнителя. Оба ответа завершаются EOT.

Такая конструкция задаёт прозрачное предпочтение краткости. Она позволяет наблюдать направление обновления DPO, но не моделирует разнообразие реальных предпочтений. Чтобы изучать сбор ответов политики, потребуется отдельный опыт с генерацией, фиксированной версией модели и независимой оценкой кандидатов.

In [ ]:
from minillm.preference import (DPOConfig, PreferencePair, freeze_reference, train_dpo)

FILLER = " Это важно и нужно учитывать при работе с моделью."

def build_pairs(n: int, seed: int = 0) -> list[PreferencePair]:
    rng = random.Random(seed)
    pairs = []
    for _ in range(n):
        question, answer = rng.choice(QUESTIONS)
        s = rng.choice(RU_SLOTS["subject"])
        v, o = rng.choice(RU_SLOTS["verb"]), rng.choice(RU_SLOTS["obj"])
        prompt_ids, _ = template.encode_with_mask(
            [Message("user", question.format(s=s))], tokenizer,
            add_generation_prompt=True)
        short = answer.format(s=s.capitalize(), v=v, o=o)
        verbose = short + FILLER * 2
        pairs.append(PreferencePair(
            prompt_ids=prompt_ids,
            chosen_ids=tokenizer.encode(short) + [EOT_ID],
            rejected_ids=tokenizer.encode(verbose) + [EOT_ID],
        ))
    return pairs

pairs = build_pairs(192, seed=SEED)
print(f"пар: {len(pairs)}")
print("chosen  :", tokenizer.decode(pairs[0].chosen_ids)[:90])
print("rejected:", tokenizer.decode(pairs[0].rejected_ids)[:90])

## 7. DPO

Замороженная копия SFT-модели задаёт референсные вероятности. DPO сравнивает изменения логарифмов вероятностей двух ответов относительно этой копии. После обучения для инференса нужна только обновлённая политика.

`margin` — средняя разность неявных наград, `accuracy` — доля пар, в которых она положительна. Ровно нулевая разность здесь не считается успехом. Эти показатели полезно читать вместе с loss и отдельными ответами. Ни значение 0,5, ни быстрое достижение 1,0 сами по себе не устанавливают причину: возможны лёгкие пары, особенности ничьих, слабое обучение, утечка или переобучение.

In [ ]:
policy = copy.deepcopy(sft_model)
reference = freeze_reference(sft_model)
dpo_state = train_dpo(policy, pairs,
                      DPOConfig(beta=0.1, lr=1e-4, steps=BUDGET["dpo_steps"],
                                batch_size=BUDGET["dpo_batch"], seed=SEED,
                                device=DEVICE, log_every=BUDGET["dpo_steps"] // 8),
                      pad_id=PAD_ID, reference=reference)

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in dpo_state.history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(steps, [h["loss"] for h in dpo_state.history], color="#5E81AC")
axes[0].axhline(np.log(2), ls="--", lw=1, color="#4C566A", label="ln 2 — старт")
axes[0].set_title("DPO-лосс"); axes[0].legend()
axes[1].plot(steps, [h["margin"] for h in dpo_state.history], color="#A3BE8C")
axes[1].axhline(0, ls="--", lw=1, color="#4C566A")
axes[1].set_title("margin: разность неявных наград")
axes[2].plot(steps, [h["chosen_reward"] for h in dpo_state.history],
             label="chosen", color="#A3BE8C")
axes[2].plot(steps, [h["rejected_reward"] for h in dpo_state.history],
             label="rejected", color="#BF616A")
axes[2].set_title("неявные награды"); axes[2].legend()
for ax in axes:
    ax.set_xlabel("шаг"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

last = dpo_state.history[-1]
print(f"margin {dpo_state.history[0]['margin']:+.3f} → {last['margin']:+.3f}, "
      f"accuracy {last['accuracy']:.2f}")

Третий график показывает два слагаемых разности. DPO может увеличивать разрыв, не увеличивая абсолютную вероятность предпочтённого ответа; неявная награда тоже не является прямой оценкой полезности. Дрейф обоих значений вниз требует анализа, но сам по себе не доказывает деградацию текста.

В наших парах длина и повтор заполнителя дают простой отличительный признак. Проверьте, насколько быстро их разделяет текущий запуск. Затем уменьшите число повторов и сравните динамику, сохранив остальные условия. Чтобы судить о качестве, добавьте отложенные пары и проверку содержания; красивый график обучения не заменяет эти измерения.

In [ ]:
question = "Для чего служит оптимизатор?"
for label, model in (("после SFT", sft_model), ("после DPO", policy)):
    answer, n = ask(model, question, max_new_tokens=80, temperature=0.7)
    print(f"--- {label} ({n} токенов)\n{answer!r}\n")
print('Два ответа — иллюстрация. Для вывода об изменении поведения сравните несколько фиксированных запросов и сохраните исходы.')

## 8. Сохранение

После DPO сохраняем **inference-артефакт**: веса, `config.json`, словарь и JSON
с происхождением стадий. Это не resumable checkpoint — DPO-веса уже не
соответствуют состояниям SFT-оптимизатора, а выдавать старые моменты за пригодные
для продолжения обучения было бы тихой ошибкой. Полный resumable checkpoint
претрейна остаётся в `runs/pretrain/final`.

In [ ]:
from minillm.train import save_model_artifact

out_dir = Path("runs/posttrain/final")
save_model_artifact(out_dir, policy, {
    "stage": "dpo",
    "sft_steps": sft_state.step,
    "dpo_steps": dpo_state.step,
    "note": "inference artifact; DPO optimizer state is intentionally not stored",
})
tokenizer.save(out_dir / "tokenizer.json")
print("сохранено:", sorted(p.name for p in out_dir.iterdir()))

import json
metrics = {"seed": SEED, "budget": BUDGET, "eot": eot_rows,
           "shared_mask_scores": shared_scores, "dpo_history": dpo_state.history}
(out_dir / "teaching_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

## 9. Итог и границы

Мы проследили, как шаблон диалога, маска целей и упаковка определяют задачу SFT. Опыт с EOT сравнивает две маски и проверяет результат по общему правилу. DPO использует заданные шаблонные предпочтения; направление margin описывает именно эти пары.

SFT способен обучать и формату, и содержанию. Обучение по предпочтениям меняет поведение в соответствии с сигналом, качество которого тоже нужно проверять. Здесь измерения относятся к синтетическим диалогам и заданным парам. Полезность реальных ответов и результаты отдельного RL-обучения потребуют собственных экспериментов и критериев оценки.

Сохраните реальные значения loss, margin, вероятности EOT и длины ответов вместе с бюджетом. Они относятся к этому запуску и этому корпусу. Далее ноутбук 3 проверяет инференс, а ноутбук 4 — контролируемые изменения архитектуры и обучения.